In [ ]:
import pandas as pd
import os

# ==========================================
# 1. Configuration and Paths
# ==========================================

# Base Paths (Uncomment the active path)
BASE_PATH = 'E:\\Projetos\\ABM-WP' # Currently active path

# Input File (Generated by previous classification scripts)
INPUT_FILENAME = 'Tabela_consumidores_Itapua_com_setor_comportamento_e_renda.csv'

# Output Files
OUTPUT_SECTOR_REPORT = 'Relatorio_Comportamento_Por_Setor.csv'
OUTPUT_GENERAL_REPORT = 'Resumo_Geral_Comportamento.csv'

def main():
    print("--- Starting Consumption Profile Analysis by Sector ---")

    # ==========================================
    # 2. Load Data
    # ==========================================
    input_path = os.path.join(BASE_PATH, 'includes', INPUT_FILENAME)
    
    if not os.path.exists(input_path):
        print(f"Error: Input file not found at {input_path}")
        return

    print(f"Reading file: {input_path}")
    df = pd.read_csv(input_path, sep=';')

    # ==========================================
    # 3. Data Cleaning
    # ==========================================
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print("\nNull values detected (before cleaning):")
        print(null_counts[null_counts > 0])
        
    # Remove rows with null values to ensure accuracy
    df_clean = df.dropna()
    print(f"Records after cleaning: {len(df_clean)}")

    # ==========================================
    # 4. Sector Analysis (Aggregation)
    # ==========================================
    
    # Aggregating data by Census Sector (CD_SETOR)
    sector_summary = df_clean.groupby('CD_SETOR').agg(
        total_consumers=('SK_MATRICULA', 'count'),
        environmentalist_count=('TP_COMPORTAMENTO', lambda x: (x == 'AMBIENTALISTA').sum()),
        moderate_count=('TP_COMPORTAMENTO', lambda x: (x == 'MODERADO').sum()),
        wasteful_count=('TP_COMPORTAMENTO', lambda x: (x == 'PERDULARIO').sum()),
        avg_daily_consumption=('NN_CONSUMO_DIARIO', 'mean')
    ).reset_index()

    # Calculate Percentages
    sector_summary['pct_environmentalist'] = (sector_summary['environmentalist_count'] / sector_summary['total_consumers']) * 100
    sector_summary['pct_moderate'] = (sector_summary['moderate_count'] / sector_summary['total_consumers']) * 100
    sector_summary['pct_wasteful'] = (sector_summary['wasteful_count'] / sector_summary['total_consumers']) * 100

    # Sort by highest percentage of Wasteful consumers (Critical for analysis)
    sector_summary = sector_summary.sort_values(by='pct_wasteful', ascending=False)

    # ==========================================
    # 5. Save Sector Report
    # ==========================================
    results_dir = os.path.join(BASE_PATH, 'resultados')
    
    # Ensure results directory exists
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)

    report_path = os.path.join(results_dir, OUTPUT_SECTOR_REPORT)
    sector_summary.to_csv(report_path, sep=';', index=False, float_format='%.2f')

    # ==========================================
    # 6. General Statistics
    # ==========================================
    general_summary = {
        'total_consumers': df_clean['SK_MATRICULA'].nunique(),
        'total_sectors': df_clean['CD_SETOR'].nunique(),
        'global_avg_daily_consumption': df_clean['NN_CONSUMO_DIARIO'].mean(),
        'global_pct_environmentalist': (df_clean['TP_COMPORTAMENTO'] == 'AMBIENTALISTA').mean() * 100,
        'global_pct_moderate': (df_clean['TP_COMPORTAMENTO'] == 'MODERADO').mean() * 100,
        'global_pct_wasteful': (df_clean['TP_COMPORTAMENTO'] == 'PERDULARIO').mean() * 100
    }

    # Save General Summary
    general_df = pd.DataFrame([general_summary])
    general_report_path = os.path.join(results_dir, OUTPUT_GENERAL_REPORT)
    general_df.to_csv(general_report_path, sep=';', index=False, float_format='%.2f')

    # ==========================================
    # 7. Output Results
    # ==========================================
    print("\n" + "="*50)
    print("ANALYSIS RESULTS")
    print("="*50)
    
    print("\nTop 5 Sectors with highest percentage of Wasteful users:")
    print(sector_summary[['CD_SETOR', 'total_consumers', 'pct_wasteful', 'avg_daily_consumption']].head())

    print("\nGeneral Statistics:")
    print(f"Total Sectors Analyzed: {len(sector_summary)}")
    print(f"Avg Consumers per Sector: {sector_summary['total_consumers'].mean():.1f}")
    
    top_sector = sector_summary.iloc[0]
    print(f"Critical Sector (Highest Waste): {top_sector['CD_SETOR']} ({top_sector['pct_wasteful']:.1f}% wasteful)")

    print("\nReports successfully saved:")
    print(f"1. {report_path}")
    print(f"2. {general_report_path}")

if __name__ == "__main__":
    main()

--- Starting Consumption Profile Analysis by Sector ---
Reading file: E:\Projetos\ABMS-WP\includes\Tabela_consumidores_Itapua_com_setor_comportamento_e_renda.csv

Null values detected (before cleaning):
NN_MEDIA_CONSUMO              527
NN_CONSUMO_DIARIO             527
VL_RENDA_MEDIA_RESPONSAVEL     35
NN_MEDIA_MORADORES_IBGE        35
dtype: int64
Records after cleaning: 14745

ANALYSIS RESULTS

Top 5 Sectors with highest percentage of Wasteful users:
   CD_SETOR  total_consumers  pct_wasteful  avg_daily_consumption
2      S002                1    100.000000           71778.688525
42     S068               91     74.725275             445.086971
34     S045               97     70.103093             347.000612
8      S008              302     68.543046             188.079731
24     S030              103     67.961165             278.619837

General Statistics:
Total Sectors Analyzed: 64
Avg Consumers per Sector: 230.4
Critical Sector (Highest Waste): S002 (100.0% wasteful)

Reports s